# Chapter 03 — Machine Learning Fundamental

Kalau di chapter 01 kamu belajar menulis program Python dari nol, dan di chapter 02 kamu belajar mengolah data tabular dengan NumPy dan Pandas, maka di chapter ini kita akan menggunakan semua itu untuk **membuat komputer belajar dari data**. Ini adalah chapter yang paling transformatif — di sinilah semua fondasi Python dan data analysis bertemu dengan dunia machine learning (ML) yang sering kamu dengar di berita dan lowongan kerja.

Tapi sebelum kamu membayangkan neural network yang mengenali wajah atau model bahasa yang menulis esai, kita perlu jujur: ML di level fundamental itu sederhana, bahkan membosankan — dan itu bagus. Justru karena sederhana, kita bisa benar-benar paham **kenapa** sebuah model bekerja, **kapan** ia gagal, dan **apa** yang harus kita waspadai saat melatihnya. Di chapter ini kita akan fokus pada empat algoritma klasik (Logistic Regression, KNN, Decision Tree, Random Forest) untuk klasifikasi, dua algoritma regresi (Linear, Ridge, Lasso), dan satu algoritma unsupervised (K-Means). Kita tidak akan menyentuh deep learning dulu — itu untuk chapter 04.

Cara paling efektif membaca chapter ini: perlakukan setiap cell sebagai percakapan. Cell markdown adalah paragraf penjelasan, cell kode adalah eksperimen kecil yang bisa kamu jalankan dan modifikasi. Kalau ada cell yang terlihat panjang, pecah jadi dua — tidak usah takut untuk menambah cell baru sendiri. Notebook ini bukan bacaan线性, ini laboratorium.

---

## Section 1: Apa Itu Machine Learning?

Bayangkan kamu bekerja di sebuah bank dan mendapat tugas menulis program yang memutuskan siapa yang layak mendapat kredit. Cara klasik adalah kamu duduk bersama manajer, menulis aturan: "jika penghasilan di atas 5 juta, tidak punya tanggungan, dan pekerjaan tetap, maka setujui". Aturan itu lalu kamu terjemahkan ke `if` dan `else`. Itu adalah **rule-based programming** — programmer menulis logikanya, komputer menjalankan.

Masalahnya, di dunia nyata aturan itu tidak pernah cukup. Bagaimana kalau ada kombinasi "penghasilan 4.9 juta, tanggungan 1, pekerjaan tetap, tapi usia di bawah 25 tahun"? Apakah layak? Aturan manusia tidak bisa mencakup semua kombinasi. Di sinilah machine learning berperan: **kita berikan data historis (penghasilan, tanggungan, pekerjaan, usia, dan keputusan: layak/tidak layak) ke komputer, lalu biarkan komputer menemukan aturannya sendiri**.

Jadi definisi formalnya: machine learning adalah kemampuan sebuah sistem untuk **belajar pola dari data** tanpa diprogram secara eksplisit. Bukan berarti programmer tidak berperan — programmer tetap memilih algoritma, membersihkan data, dan mengevaluasi hasil. Yang berbeda adalah **aturan final** yang menghasilkan prediksi, itu ditemukan oleh algoritma dari data, bukan ditulis tangan.

Di dunia ML ada tiga paradigma besar. Yang pertama adalah **supervised learning** — kita punya data dengan label (jawaban benar sudah tersedia), dan kita minta model belajar memetakan input ke label. Contoh: klasifikasi email spam/bukan spam, regresi harga rumah. Yang kedua adalah ** unsupervised learning ** — kita punya data tanpa label, dan kita minta model menemukan struktur di dalamnya. Contoh: segmentasi pelanggan, deteksi anomali. Yang ketiga adalah **reinforcement learning** — agen belajar dengan mencoba-coba dan mendapat reward/penalty (game AI, robot). Di chapter ini kita akan fokus ke supervised learning (klasifikasi dan regresi) dan menyinggung unsupervised (K-Means, PCA) di section 10.

Mari kita lihat sebuah contoh kecil. Bayangkan kamu punya data 4 mahasiswa: nama, jurusan, dan IPK. Tugasmu adalah memprediksi IPK mahasiswa baru berdasarkan jurusannya. Ini adalah masalah regresi (memprediksi angka). Untuk menyederhanakan, kita buat data sintetis dan lihat seperti apa supervised learning itu secara konkret.

In [ ]:
# Contoh supervised learning: prediksi IPK dari jurusan + semester
import pandas as pd
import numpy as np

data = {
    'jurusan'  : ['IF', 'IF', 'SI', 'SI', 'TK', 'TK', 'IF', 'SI'],
    'semester' : [2, 4, 3, 5, 2, 6, 8, 4],
    'ipk'      : [3.2, 3.6, 3.4, 3.5, 3.1, 3.3, 3.8, 3.5]
}
df = pd.DataFrame(data)
print(df)
print(f"\nRata-rata IPK per jurusan:")
print(df.groupby('jurusan')['ipk'].mean())

Apa yang baru saja kita lakukan? Kita menghitung rata-rata IPK per jurusan — itu adalah bentuk paling sederhana dari supervised learning: **modelnya adalah sebuah lookup table** (jurusan → rata-rata IPK). Untuk data baru "mahasiswa SI semester 3", model akan menjawab 3.45 (rata-rata SI). Algoritma ML yang lebih canggih (linear regression, decision tree) pada dasarnya melakukan hal serupa, tapi dengan kemampuan menangkap pola yang lebih kompleks.

Yang penting untuk dipahami sekarang: di supervised learning kita punya dua hal — **X (fitur, input)** dan **y (label, target)**. Model kita akan belajar fungsi `f(X) ≈ y`. Kata "latih" atau "train" dalam ML berarti **mengubah parameter-parameter model** agar `f(X)` sedekat mungkin dengan `y` pada data yang kita punya.

In [ ]:
# Pisahkan X (fitur) dan y (target) — ini konvensi universal di scikit-learn
X = df[['jurusan', 'semester']]  # fitur: 2D
y = df['ipk']                     # target: 1D

print("X (fitur):")
print(X.head())
print(f"\nX adalah {type(X).__name__} dengan shape {X.shape}")
print(f"y adalah {type(y).__name__} dengan shape {y.shape}")

Perhatikan baik-baik output di atas: `X` selalu **2D** (DataFrame atau 2D array) dengan bentuk `(n_samples, n_features)`, sedangkan `y` selalu **1D** (Series atau 1D array) dengan panjang `n_samples`. Ini adalah konvensi yang dipakai di seluruh scikit-learn, dan kalau kamu melanggarnya (misal X 1D), hampir semua fungsi akan error. Ini salah satu hal pertama yang harus dihafal: **X 2D, y 1D**.

Juga, `n_samples` artinya jumlah baris (jumlah observasi), dan `n_features` artinya jumlah kolom (jumlah variabel input). Dalam contoh kita, `n_samples=8` dan `n_features=2` (jurusan + semester).

---

**Mini-check refleksi 1:**

1. Apa beda utama rule-based programming dengan machine learning? Coba jelaskan dengan kata-katamu sendiri, jangan copy paste definisi.
2. Kapan supervised learning tepat dipakai, dan kapan unsupervised? Beri satu contoh kasus untuk masing-masing yang **berbeda** dari contoh di atas.
3. Mengapa `X` di scikit-learn selalu 2D sementara `y` selalu 1D? Apa yang akan terjadi kalau kamu secara tidak sengaja passing `X` 1D ke sebuah classifier?

---

## Section 2: Alur Kerja ML & Train/Test Split

Sekarang kamu sudah tahu bahwa ML adalah soal belajar pola dari data. Tapi prosesnya tidak langsung "kasih data, jadi model". Ada alur yang harus diikuti, dan di industri alur ini distandarkan dalam sebuah metodologi bernama **CRISP-DM** (Cross-Industry Standard Process for Data Mining). Alurnya adalah: Business Understanding → Data Understanding → Data Preparation → Modeling → Evaluation → Deployment, dengan loop iterasi di antara langkah-langkah tersebut.

Di chapter ini kita tidak akan membahas semua langkah CRISP-DM secara mendalam (itu materi untuk data engineer profesional), tapi kita akan fokus pada **satu keputusan paling kritis** yang membedakan ML yang benar dari ML yang sia-sia: **train/test split**.

Begini analoginya. Bayangkan kamu sedang belajar untuk ujian. Ada dua cara belajar. Cara pertama: kamu menghafal **semua** soal latihan dan jawabannya. Saat ujian, ternyata soalnya berbeda, kamu bingung. Cara kedua: kamu belajar dari **sebagian** soal latihan, lalu **menyisakan sebagian kecil** yang belum pernah kamu lihat untuk menguji dirimu sendiri. Cara kedua jelas lebih jujur — itu yang akan kita lakukan di ML.

Train/test split artinya: sebelum melatih model, kita acak dan bagi data menjadi dua bagian. **Training set** (biasanya 70-80% data) dipakai untuk melatih model. **Test set** (sisanya 20-30%) disimpan rapat-rapat, tidak dipakai untuk melatih, hanya dipakai di akhir untuk mengukur seberapa bagus model pada data yang belum pernah ia lihat. Tanpa split ini, kita tidak akan pernah tahu apakah model kita **benar-benar** memahami pola, atau hanya menghafal data.

Mari kita lakukan split pada dataset Iris. Ini dataset klasik yang akan kamu temui di hampir setiap tutorial ML — 150 bunga Iris dengan 4 fitur (panjang/lebar sepal dan petal) dan 3 kelas (setosa, versicolor, virginica).

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

iris = load_iris()
X = iris.data       # (150, 4) — 150 bunga, 4 fitur
y = iris.target     # (150,)  — label 0, 1, 2

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,    # 20% test, 80% train
    random_state=42,  # reproducibility
    stratify=y        # preserve proporsi kelas
)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Distribusi kelas di train: {dict(zip(*np.unique(y_train, return_counts=True)))}")
print(f"Distribusi kelas di test : {dict(zip(*np.unique(y_test, return_counts=True)))}")

Perhatikan tiga argumen penting. `test_size=0.2` artinya 20% data masuk ke test, 80% ke train. `random_state=42` adalah **benih keacakan** — tanpa argumen ini, setiap kali kamu run cell, hasil split akan berbeda. Dengan `random_state=42`, kamu dijamin mendapat split yang sama persis setiap kali. Ini penting untuk **reproduktifitas** — saat kamu debugging atau membandingkan model, kamu ingin hasilnya konsisten.

Argumen ketiga, `stratify=y`, adalah yang paling halus. Pada output di atas, baik train maupun test punya proporsi kelas yang sama (masing-masing 40, 40, 40 di train dan 10, 10, 10 di test, atau 33%/33%/33% proporsional). Tanpa `stratify`, bisa saja test set kamu kebetulan hanya berisi 2 kelas dan kehilangan 1 kelas sama sekali — model kamu tidak akan pernah bisa dievaluasi dengan benar untuk kelas yang hilang itu.

Perhatikan juga bahwa kita **tidak pernah menggunakan `X_test` atau `y_test` sampai model selesai dilatih**. Itu aturan yang sangat ketat dan akan kita lihat konsekuensinya di section berikutnya.

In [ ]:
# Eksperimen: split dengan dan tanpa stratify
X_train_strat, X_test_strat, y_train_strat, y_test_strat = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train_no, X_test_no, y_train_no, y_test_no = train_test_split(
    X, y, test_size=0.2, random_state=42  # tanpa stratify
)

print("Dengan stratify — kelas di test:", np.unique(y_test_strat, return_counts=True))
print("Tanpa stratify  — kelas di test:", np.unique(y_test_no, return_counts=True))

Pada dataset yang seimbang (seperti Iris) perbedaan ini kadang tidak terlalu terasa. Tapi pada dataset yang **tidak seimbang** (misalnya 95% transaksi normal, 5% fraud), tanpa `stratify` test set kamu bisa kebetulan berisi 100% transaksi normal — model yang selalu memprediksi "normal" akan mendapat akurasi 100% di test, tapi itu bohong. **Selalu gunakan `stratify` untuk klasifikasi** kecuali kamu tahu proporsi kelas di train dan test harus berbeda (kasus yang sangat jarang).

---

**Mini-check refleksi 2:**

1. Mengapa `random_state=42` penting untuk reproduktifitas? Apa konsekuensinya kalau kamu tidak menyertakannya?
2. Kapan `stratify=y` wajib dipakai, dan kapan boleh diabaikan? Beri satu skenario di mana pengabaiannya akan menyebabkan evaluasi model menyesatkan.
3. Setelah split dilakukan, sampai kapan `X_test` boleh kamu "sentuh" dengan kode apa pun selain `predict` di akhir? Jelaskan aturan mainnya.

---

## Section 3: Preprocessing — Scaling, Encoding, Pipeline

Data mentah hampir tidak pernah bisa langsung masuk ke algoritma ML. Ada tiga preprocessing yang akan kamu temui di 80% project ML: **scaling** (mengubah skala fitur numerik), **encoding** (mengubah data kategorik menjadi numerik), dan **pipeline** (mengikat preprocessing + model jadi satu objek yang aman dari data leakage).

Kita mulai dari scaling. Bayangkan kamu punya dua fitur: usia (rentang 0-100) dan penghasilan (rentang 0-jutaan). Algoritma seperti logistic regression, KNN, dan SVM menghitung **jarak** antar titik data. Titik yang berbeda 1 tahun usia dan titik yang berbeda Rp 1.000.000 penghasilan akan diperlakukan sangat berbeda — padahal "jarak 1 tahun" di skala manusia mungkin sama signifikannya dengan "jarak Rp 1.000.000". Scaling menyamakan skala sehingga semua fitur diperlakukan secara adil.

Tiga scaler utama di scikit-learn. **StandardScaler** mengubah data sehingga mean = 0 dan standar deviasi = 1 — cocok untuk data yang berdistribusi normal. **MinMaxScaler** mengubah data ke rentang [0, 1] — cocok untuk neural network atau data yang tidak berdistribusi normal. **RobustScaler** menggunakan median dan IQR, lebih tahan terhadap outlier — cocok untuk data dengan pencilan.

Mari kita lihat bagaimana scaling bekerja dalam praktik. Kita akan pakai dataset Iris yang sudah kita split.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)   # ← PENTING: hanya transform, bukan fit

print("Sebelum scaling (X_train[0]):", X_train[0])
print("Setelah scaling (X_train_scaled[0]):", X_train_scaled[0].round(3))
print(f"\nMean per fitur (setelah scaling): {X_train_scaled.mean(axis=0).round(3)}")
print(f"Std  per fitur (setelah scaling): {X_train_scaled.std(axis=0).round(3)}")

Perhatikan baris `scaler.transform(X_test)` — di sini kita **tidak** memanggil `fit_transform`, hanya `transform`. Ini aturan paling penting di preprocessing: scaler (atau encoder) di-**fit** hanya pada training set, lalu dipakai untuk **transform** test set. Kenapa? Karena kalau kamu fit scaler pada seluruh data (train + test), informasi dari test set akan "bocor" ke scaler, dan secara tidak langsung ke model. Ini namanya **data leakage** — dan itu bug ML paling berbahaya karena model terlihat bagus di test tapi gagal total di production.

Verifikasi: setelah scaling, `mean` mendekati 0 dan `std` mendekati 1. Itu tanda scaling bekerja dengan benar. Kalau kamu cek `X_test_scaled.mean()`, nilainya **tidak** persis 0 — dan itu normal, karena mean dan std dihitung dari train, dan test set pasti sedikit berbeda.

In [ ]:
# Eksperimen: apa yang terjadi kalau kita fit pada seluruh data (data leakage)
scaler_bad = StandardScaler()
X_all_scaled_bad = scaler_bad.fit_transform(X)   # fit pada SEMUA data — salah

scaler_good = StandardScaler()
X_train_good = scaler_good.fit_transform(X_train)
X_test_good  = scaler_good.transform(X_test)     # hanya transform — benar

print(f"Test set scaled (good practice): mean={X_test_good.mean():.4f}, std={X_test_good.std():.4f}")
print("Test set tidak persis mean 0 — dan itu benar, karena mean dihitung dari train.")

Sekarang kita masuk ke encoding. Algoritma ML bekerja dengan angka, bukan string. Kalau kolom `jurusan` berisi `['IF', 'SI', 'TK']`, kita harus mengubahnya jadi angka. Ada dua encoder utama. **LabelEncoder** mengubah kategori jadi integer ordinal — `IF=0, SI=1, TK=2`. Tapi ini berbahaya untuk data nominal (tanpa urutan), karena algoritma akan mengira TK "lebih besar" dari IF. **OneHotEncoder** menghindari jebakan ini dengan membuat satu kolom biner per kategori — `jurusan_IF`, `jurusan_SI`, `jurusan_TK` — sehingga tidak ada urutan tersirat. Untuk kolom dengan banyak kategori (ratusan), one-hot bisa menjadi masalah (curse of dimensionality) dan biasanya dipakai teknik lain seperti target encoding.

In [ ]:
from sklearn.preprocessing import OneHotEncoder
import pandas as pd

df_jurusan = pd.DataFrame({'jurusan': ['IF', 'SI', 'TK', 'IF', 'SI']})
ohe = OneHotEncoder(sparse_output=False)
encoded = ohe.fit_transform(df_jurusan)

print("Kategori asli:", df_jurusan['jurusan'].tolist())
print("Encoded:\n", encoded)
print("Nama kolom baru:", ohe.get_feature_names_out(['jurusan']))

Terakhir, **Pipeline** dan **ColumnTransformer** adalah dua alat paling penting untuk mencegah data leakage secara sistematis. **Pipeline** mengikat beberapa langkah (misal: scaler → classifier) jadi satu objek. Saat kamu panggil `pipeline.fit(X_train, y_train)`, semua langkah internal dilakukan dengan benar: scaler di-fit pada `X_train`, classifier dilatih pada `X_train_scaled`. Saat kamu panggil `pipeline.predict(X_test)`, scaler hanya `transform` (bukan `fit_transform`) pada `X_test`. Jadi **kamu tidak bisa lupa** untuk tidak menyentuh test set, karena pipeline yang akan mengurusnya.

**ColumnTransformer** memungkinkan kita menerapkan preprocessing berbeda pada kolom berbeda dalam satu pipeline. Misalnya, kolom numerik discaling dengan `StandardScaler`, kolom kategorik di-encode dengan `OneHotEncoder` — keduanya sekaligus.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Untuk Iris, semua fitur numerik jadi satu blok
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])

pipeline.fit(X_train, y_train)
acc = pipeline.score(X_test, y_test)
print(f"Akurasi pada test set: {acc:.4f}")
print(f"Langkah pipeline: {[n for n, _ in pipeline.steps]}")

Lihat betapa sederhananya: satu `fit`, satu `score`. Pipeline mengurus semua preprocessing dengan benar secara internal. Kalau di kemudian hari kamu ingin ganti classifier (misal dari LogisticRegression ke RandomForest), kamu hanya perlu ganti satu baris dalam definisi pipeline — kode pelatihan dan evaluasi tetap sama. Inilah kekuatan abstraksi di scikit-learn.

Tapi perlu dicatat: ColumnTransformer baru benar-benar terasa powerful kalau kamu punya data tabular campuran (numerik + kategorik). Untuk Iris yang semuanya numerik, Pipeline biasa sudah cukup. Nanti di chapter 05 (ML Advanced) dan project kita akan sering pakai ColumnTransformer.

---

**Mini-check refleksi 3:**

1. Jelaskan dengan analogi dapur atau jalan raya mengapa `fit_transform` pada test set adalah data leakage. Kenapa hal ini bisa bikin model "terlihat bagus" di test tapi gagal di production?
2. Kapan `LabelEncoder` boleh dipakai, dan kapan harus diganti `OneHotEncoder`? Beri contoh kasus untuk masing-masing.
3. Kalau kamu punya dataset dengan 5 kolom numerik dan 2 kolom kategorik, apa keuntungan menggunakan `ColumnTransformer` dibanding melakukan scaling dan encoding secara manual sebelum melatih model?

---

## Section 4: Klasifikasi — Logistic Regression

Sekarang kita masuk ke algoritma pertama yang akan kita pelajari secara mendalam: **Logistic Regression**. Jangan tertipu namanya — meski disebut "regression", algoritma ini adalah classifier. Nama "regression" di sini merujuk pada transformasi matematis di dalamnya, bukan tugas yang dilakukan.

Logistic Regression bekerja dengan cara yang elegan. Pertama, ia menghitung skor linear `z = w·x + b` (sama persis dengan linear regression). Lalu ia mengubah skor `z` itu menjadi **probabilitas** dengan fungsi **sigmoid**:

```
p = σ(z) = 1 / (1 + e^(-z))
```

Sigmoid adalah kurva S yang mengubah semua bilangan real ke rentang (0, 1). Kalau `z` sangat negatif, `p` mendekati 0. Kalau `z` sangat positif, `p` mendekati 1. Kalau `z = 0`, `p = 0.5` — titik tengah, kebimbangan. Untuk klasifikasi biner, kita putuskan kelas 1 kalau `p > 0.5`, kelas 0 sebaliknya.

Untuk klasifikasi multi-kelas (seperti Iris dengan 3 spesies), Logistic Regression melatih **satu model per kelas** (satu-vs-rest), lalu memilih kelas dengan probabilitas tertinggi. Itulah kenapa output dari `predict_proba` adalah sebuah matriks — satu kolom probabilitas per kelas.

Mari kita latih Logistic Regression pada data Iris. Kita pakai pipeline yang sudah kita bangun untuk memastikan scaling dilakukan dengan benar.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

model = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)

print("Prediksi 5 sampel pertama:", y_pred[:5])
print("Probabilitas 5 sampel pertama (per kelas):\n", y_proba[:5].round(3))

Perhatikan `predict_proba` mengembalikan **probabilitas** untuk setiap kelas, dan `predict` mengembalikan **kelas dengan probabilitas tertinggi**. Untuk sampel pertama, model yakin 100% itu kelas 0 (setosa). Untuk sampel kedua, model sedikit ragu: 88% kelas 1, 11% kelas 2 — tapi tetap memilih kelas 1. Probabilitas seperti ini sangat berguna di industri: kalau threshold keakuratan 0.9 tidak tercapai, kamu bisa menahan prediksi dan minta review manusia.

Logistic Regression punya satu keunggulan interpretasi yang tidak dimiliki model lain: koefisiennya (`model.coef_`) bisa dibaca langsung. Tiap fitur punya satu koefisien per kelas. Koefisien positif artinya fitur meningkatkan probabilitas kelas tersebut, negatif menurunkan.

In [ ]:
# Akses koefisien dari classifier (langkah kedua di pipeline)
clf = model.named_steps['clf']
print("Bentuk koefisien:", clf.coef_.shape)  # (3 kelas, 4 fitur)
print("\nKoefisien per kelas (baris = kelas, kolom = fitur):")
print(pd.DataFrame(clf.coef_, columns=iris.feature_names).round(2))
print("\nIntercept per kelas:", clf.intercept_.round(2))

Bisa kamu baca sendiri: untuk kelas 0 (setosa), fitur `petal length` punya koefisien -1.51 (sangat negatif) — artinya semakin panjang petal, semakin rendah probabilitas kelas 0. Untuk kelas 2 (virginica), koefisien `petal length` positif (1.61) — masuk akal, virginica punya petal paling panjang. Ini salah satu alasan Logistic Regression tetap populer di industri: **bisa dijelaskan ke stakeholder non-teknis**.

Logistic Regression punya keterbatasan. Ia bekerja paling baik ketika batas keputusan (decision boundary) antar kelas bisa digambar sebagai **garis lurus** (atau hyperplane di dimensi tinggi). Kalau data kamu tidak linearly separable — misalnya bentuk bulan sabit atau dua spiral yang saling melilit — Logistic Regression akan kesulitan. Untuk kasus seperti itu, kita butuh algoritma lain (KNN, decision tree, atau neural network).

---

**Mini-check refleksi 4:**

1. Logistic Regression dinamai "regression" tapi dipakai untuk klasifikasi. Dari transformasi matematisnya, jelaskan kenapa nama itu masuk akal.
2. Apa beda `predict` dan `predict_proba`? Kapan kamu lebih butuh yang satu dibanding yang lain?
3. Kalau koefisien Logistic Regression untuk fitur `pengalaman_kerja` pada kelas "layak kredit" adalah -0.8, apa artinya secara bisnis? Apakah pengalaman kerja yang lebih lama menurunkan kemungkinan seseorang dianggap layak?

---

## Section 5: Klasifikasi — K-Nearest Neighbors (kNN)

K-Nearest Neighbors adalah algoritma yang paling intuitif di ML. Ia tidak punya "pelatihan" dalam arti tradisional — saat kamu panggil `fit(X, y)`, ia **hanya menyimpan** data. Yang bekerja adalah saat prediksi: untuk titik baru, cari K tetangga terdekat di data training, lalu voting — kelas mayoritas dari K tetangga itu yang jadi prediksi.

Analogi yang paling cocok: pikirkan sebuah kantor notaris. Saat ada kasus baru yang mirip dengan kasus sebelumnya, notaris bertanya ke rekan-rekannya: "dari K kasus terakhir yang paling mirip dengan kasus ini, bagaimana kalian menyelesaikannya?" Mayoritas jawaban jadi pegangan. Tidak ada "model" yang dilatih — hanya memori kasus lalu dan voting.

Pemilihan K adalah keputusan paling kritis dalam kNN. K=1 artinya prediksi hanya berdasarkan satu tetangga terdekat — sangat sensitif terhadap noise. K terlalu besar (misal K = jumlah data) artinya voting terlalu banyak, hasilnya selalu kelas mayoritas. Untuk dataset kecil, K antara 3 dan 11 biasanya titik awal yang baik. Cara paling andal memilih K adalah dengan **cross-validation** — kita coba beberapa nilai K, hitung akurasi rata-rata, pilih K dengan akurasi tertinggi.

Mari kita latih kNN dengan K=5 dan lihat akurasinya.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

knn = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', KNeighborsClassifier(n_neighbors=5))
])
knn.fit(X_train, y_train)
print(f"Akurasi KNN (K=5): {knn.score(X_test, y_test):.4f}")

Sekarang mari kita coba berbagai nilai K dan lihat mana yang terbaik. Kita pakai **cross-validation** pada training set, bukan test set — karena kalau kita tuning K berdasarkan akurasi test, kita akan "menyentuh" test set sebelum waktunya (indirect overfitting). Konsep cross-validation akan kita bahas lebih detail di section 11.

In [ ]:
from sklearn.model_selection import cross_val_score
import numpy as np

k_range = range(1, 21)
cv_scores = []

for k in k_range:
    knn_tmp = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', KNeighborsClassifier(n_neighbors=k))
    ])
    scores = cross_val_score(knn_tmp, X_train, y_train, cv=5)
    cv_scores.append(scores.mean())

best_k = k_range[np.argmax(cv_scores)]
print(f"K terbaik dari CV: {best_k} (CV akurasi: {max(cv_scores):.4f})")

In [ ]:
# Visualisasi K vs CV akurasi
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 4))
plt.plot(k_range, cv_scores, marker='o')
plt.axvline(best_k, color='red', linestyle='--', alpha=0.5, label=f'best K = {best_k}')
plt.xlabel('K')
plt.ylabel('CV Accuracy')
plt.title('K vs Akurasi Cross-Validation')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Plot di atas adalah pola yang akan kamu temui berulang kali di ML: ada **sweet spot** K yang optimal. K terlalu kecil, model terlalu sensitif terhadap outlier di training (overfitting). K terlalu besar, model terlalu general dan kehilangan pola lokal (underfitting). Untuk Iris, K sekitar 5 sampai 11 bekerja sama baiknya — dataset ini memang mudah.

KNN juga punya parameter `weights`. Defaultnya `'uniform'` — semua tetangga punya suara sama. `'distance'` artinya tetangga lebih dekat punya suara lebih besar. Untuk dataset dengan noise, weighted KNN sering lebih stabil. Kamu bisa coba ganti `weights='distance'` di Pipeline di atas dan amati perubahan akurasinya.

In [ ]:
# Eksperimen: uniform vs distance weighting
for weights in ['uniform', 'distance']:
    knn_w = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', KNeighborsClassifier(n_neighbors=5, weights=weights))
    ])
    knn_w.fit(X_train, y_train)
    print(f"weights={weights}: test acc = {knn_w.score(X_test, y_test):.4f}")

KNN punya satu kelemahan besar yang perlu kamu tahu: **waktu prediksi**. Untuk satu titik baru, kNN harus menghitung jarak ke **semua** titik training. Kalau training set kamu 1 juta baris, prediksi satu titik akan lambat. Algoritma seperti Logistic Regression atau Random Forest jauh lebih cepat di prediksi, karena mereka sudah "meringkas" data training jadi parameter-parameter kecil saat `fit`. KNN tetap berguna untuk dataset kecil, prototyping cepat, dan baseline — tapi jarang dipakai di production untuk data besar.

---

**Mini-check refleksi 5:**

1. Mengapa kNN butuh scaling fitur, sementara Logistic Regression (setelah scaling) juga butuh tapi efeknya tidak seekstrem kNN? Pikirkan dari rumus jarak yang dipakai.
2. Apa beda K=1 dengan K=N (semua data)? Jelaskan dalam konteks voting dan underfitting/overfitting.
3. Mengapa tuning K berdasarkan akurasi test set adalah "cheating"? Bagaimana cross-validation menghindarinya?

---

## Section 6: Decision Tree & Random Forest

Decision Tree adalah algoritma yang paling mudah dijelaskan ke non-teknisi karena bekerja seperti flowchart. Algoritma ini membagi data secara bertahap dengan mengajukan pertanyaan ya/tidak pada fitur: "apakah petal length <= 2.45?" → ya → setosa. Tidak → lanjut ke pertanyaan berikutnya. Setiap "pertanyaan" di sebut **split**, dan di akhir setiap cabang ada **leaf** yang berisi kelas mayoritas.

Pertanyaan yang diajukan Tree tidak dipilih secara acak — Tree memilih split yang paling "memisahkan" kelas. Ukuran yang dipakai untuk itu adalah **Gini impurity** (atau entropy sebagai alternatif). Gini mengukur seberapa campur suatu kelompok: 0 artinya murni (semua satu kelas), 0.5 artinya campur rata (50-50). Tree memilih split yang **menjumlahkan Gini paling rendah** di kedua sisi — itulah split terbaik untuk data itu.

Decision Tree punya satu kelemahan fatal: ia **sangat mudah overfitting**. Kalau kamu biarkan Tree tumbuh tanpa batas, ia akan menghafal setiap titik data training dan mendapat akurasi sempurna di training, tapi performanya buruk di test. Cara mengatasi: batasi **max_depth** (kedalaman maksimum), **min_samples_split** (minimum sampel untuk bisa split lagi), dan **min_samples_leaf** (minimum sampel di leaf). Tuning hyperparameter ini adalah keterampilan inti dalam ML.

Mari kita latih Decision Tree dengan max_depth dibatasi dan visualisasikan strukturnya.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(max_depth=3, random_state=42)
dt.fit(X_train, y_train)
print(f"Akurasi DT (max_depth=3): {dt.score(X_test, y_test):.4f}")
print(f"Jumlah leaf: {dt.get_n_leaves()}, Kedalaman: {dt.get_depth()}")

In [ ]:
# Visualisasi tree
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 6))
plot_tree(dt, feature_names=iris.feature_names,
          class_names=iris.target_names, filled=True, rounded=True, fontsize=9)
plt.title('Decision Tree untuk Iris (max_depth=3)')
plt.tight_layout()
plt.show()

Bisa kamu baca sendiri: tiap node menampilkan fitur yang dipakai untuk split, nilai threshold, Gini impurity, jumlah sampel, dan distribusi kelas. Bisa dilihat: hanya butuh 3 pertanyaan untuk memisahkan 150 bunga Iris dengan akurasi tinggi. Petal length dan petal width adalah fitur yang paling membedakan.

Tapi akurasi DT tunggal tidak selalu bagus. Coba naikkan `max_depth=None` (tidak ada batas) dan amati apa yang terjadi: akurasi training bisa 100%, tapi akurasi test akan turun. Itulah overfitting yang tadi kita sebutkan.

In [ ]:
# Demonstrasi overfitting: tanpa batas vs dengan batas
dt_unbounded = DecisionTreeClassifier(random_state=42)
dt_unbounded.fit(X_train, y_train)

print("Tanpa batas:")
print(f"  Train acc: {dt_unbounded.score(X_train, y_train):.4f}")
print(f"  Test  acc: {dt_unbounded.score(X_test, y_test):.4f}")
print(f"  Kedalaman: {dt_unbounded.get_depth()}, Leaves: {dt_unbounded.get_n_leaves()}")

print("\nmax_depth=3:")
print(f"  Train acc: {dt.score(X_train, y_train):.4f}")
print(f"  Test  acc: {dt.score(X_test, y_test):.4f}")

Lihat perbedaannya. Tree tanpa batas mendapat train accuracy 100% (hafal) tapi test accuracy-nya tidak lebih baik dari Tree dengan max_depth=3. Itu klasik overfitting: model menghafal noise di training dan gagal menggeneralisasi.

Sekarang, **Random Forest** adalah solusi elegan untuk masalah overfitting Tree. Idealnya sederhana: **train banyak Decision Tree, lalu voting**. Setiap Tree dilatih pada subset data acak (bootstrap sampling) dan subset fitur acak. Karena Tree-Tree itu berbeda-beda, mereka akan menghafal pola yang berbeda, dan **kesalahan mereka cenderung tidak berkorelasi** — saat di-vote, error individu saling meniadakan.

Random Forest adalah salah satu algoritma **paling reliable** di ML — hampir selalu jadi baseline yang kuat. Ia butuh sedikit tuning, tahan terhadap overfitting (jika jumlah Tree cukup banyak), dan bisa menangani data campuran (numerik + kategorik) dengan preprocessing minimal. Satu-satunya kelemahan: interpretasi lebih sulit dari single Tree (kita tidak bisa visualisasikan 100 Tree sekaligus).

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
print(f"Akurasi RF (100 trees, max_depth=5): {rf.score(X_test, y_test):.4f}")

# Feature importance — fitur mana yang paling berkontribusi
import pandas as pd
imp = pd.DataFrame({
    'fitur'      : iris.feature_names,
    'importance' : rf.feature_importances_
}).sort_values('importance', ascending=False)
print("\nFeature importance:")
print(imp)

In [ ]:
# Visualisasi feature importance
plt.figure(figsize=(8, 4))
plt.barh(imp['fitur'], imp['importance'], color='steelblue')
plt.xlabel('Importance')
plt.title('Feature Importance — Random Forest')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

Untuk Iris, **petal length** dan **petal width** jauh lebih penting dari sepal measurements. Ini masuk akal dengan intuisi botanis: petal adalah bagian yang paling bervariasi antar spesies. Feature importance seperti ini sangat berguna di industri: dari 100 fitur, kamu bisa tahu 5 yang paling penting dan fokus mengumpulkan data tambahan untuk fitur itu saja.

Random Forest juga punya parameter `oob_score` (out-of-bag score) yang unik: karena setiap Tree hanya melihat ~63% data (bootstrap sampling), sisa 37% bisa dipakai sebagai validation **internal** — tanpa perlu split manual. Sangat berguna untuk monitoring saat training.

---

**Mini-check refleksi 6:**

1. Decision Tree tanpa batas bisa menghafal training set 100%. Kenapa itu masalah, padahal akurasinya "sempurna"?
2. Random Forest melatih banyak Tree yang berbeda. Apa yang menyebabkan Tree-Tree itu berbeda satu sama lain, dan kenapa itu membantu menghindari overfitting?
3. Kamu punya dataset dengan 50 fitur. Random Forest memberitahu 5 fitur terpenting. Apa implikasi praktisnya untuk pengumpulan data di masa depan?

---

## Section 7: Regresi — Linear, Ridge, Lasso

Sekarang kita geser dari klasifikasi (prediksi kategori) ke regresi (prediksi angka). Contoh regresi: prediksi harga rumah dari fitur (luas, lokasi, jumlah kamar), prediksi gaji dari pengalaman, prediksi suhu dari waktu. Algoritma yang kita pakai berbeda dari klasifikasi — outputnya angka kontinu, bukan kelas diskrit.

Algoritma paling dasar adalah **Linear Regression**. Ia mengasumsikan hubungan antara fitur (X) dan target (y) bisa digambar sebagai **garis lurus (atau hyperplane di dimensi tinggi)**: `y = w·x + b`. Algoritma ini mencari `w` dan `b` yang **meminimalkan jumlah kuadrat error** — yaitu selisih antara prediksi `ŷ` dan nilai sebenarnya `y`, dipangkatkan dua, lalu dijumlahkan. Kenapa kuadrat? Karena error positif dan negatif harus diperlakukan sama, dan kuadrat membuat fungsi loss-nya "bagus" secara matematis (turunan bisa dihitung analitik).

Linear Regression punya satu masalah besar: kalau fitur sangat banyak atau saling berkorelasi, koefisien `w` bisa meledak — nilainya sangat besar, dan model jadi sangat sensitif terhadap noise kecil di input. Hasilnya: model bekerja sangat baik di training, gagal di test (overfitting lagi). Solusinya: **regularisasi** — tambahkan penalti pada koefisien besar di fungsi loss.

Mari kita buat data regresi sintetis dan latih Linear Regression.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.datasets import make_regression

# Generate data sintetis: y = 3*x + noise
X_reg, y_reg = make_regression(n_samples=100, n_features=1, noise=15, random_state=42)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

lin = LinearRegression()
lin.fit(X_train_r, y_train_r)
print(f"Slope (w):     {lin.coef_[0]:.3f}")
print(f"Intercept (b): {lin.intercept_:.3f}")
print(f"R² pada test:   {lin.score(X_test_r, y_test_r):.4f}")

Tiga metrik regresi yang paling umum dipakai. **MAE (Mean Absolute Error)** adalah rata-rata selisih absolut antara prediksi dan aktual — mudah diinterpretasi ("rata-rata prediksi kita meleset 5 dolar dari harga sebenarnya"). **MSE (Mean Squared Error)** adalah rata-rata kuadrat selisih — menghukum error besar lebih keras. **RMSE (Root MSE)** adalah akar MSE, satuannya kembali sama dengan target, lebih intuitif dari MSE. **R²** adalah proporsi variansi target yang dijelaskan model — 1.0 artinya sempurna, 0.0 artinya model sama bodohnya dengan memprediksi rata-rata, negatif artinya model lebih buruk dari rata-rata.

R² yang kita dapat di atas (0.94) artinya model menjelaskan 94% variansi harga — cukup bagus untuk data sintetis. Di dunia nyata, R² 0.7-0.8 sudah dianggap sangat baik.

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

y_pred_r = lin.predict(X_test_r)
mse  = mean_squared_error(y_test_r, y_pred_r)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test_r, y_pred_r)
r2   = r2_score(y_test_r, y_pred_r)

print(f"MSE : {mse:.3f}")
print(f"RMSE: {rmse:.3f}  ← satuan sama dengan y")
print(f"MAE : {mae:.3f}")
print(f"R²  : {r2:.4f}")

Sekarang regularisasi. **Ridge regression (L2)** menambahkan penalti `λ × Σ w²` ke fungsi loss — efeknya, koefisien yang besar "ditarik" mendekati nol, tapi tidak pernah sampai nol. **Lasso regression (L1)** menambahkan penalti `λ × Σ |w|` — efeknya, beberapa koefisien justru ditarik **persis ke nol**. Lasso secara otomatis melakukan **feature selection**: fitur yang tidak relevan koefisiennya nol, hilang dari model. Ridge menjaga semua fitur tapi mengecilkan koefisiennya.

Kapan pakai yang mana? Ridge adalah default yang aman — terutama kalau semua fitur kamu yakini relevan. Lasso dipakai kalau kamu curiga banyak fitur tidak relevan dan ingin model yang sparse (hanya pakai sedikit fitur). Ada juga **ElasticNet** yang menggabungkan L1 + L2, tapi itu untuk chapter 04.

In [ ]:
from sklearn.linear_model import Ridge, Lasso

# Dataset dengan banyak fitur, hanya sedikit yang relevan
X_multi, y_multi = make_regression(n_samples=100, n_features=20, n_informative=5, noise=10, random_state=42)
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(X_multi, y_multi, test_size=0.2, random_state=42)

ridge = Ridge(alpha=1.0).fit(X_train_m, y_train_m)
lasso = Lasso(alpha=1.0).fit(X_train_m, y_train_m)

print(f"Ridge — R²: {ridge.score(X_test_m, y_test_m):.3f}, koefisien non-zero: {(ridge.coef_ != 0).sum()}/20")
print(f"Lasso — R²: {lasso.score(X_test_m, y_test_m):.3f}, koefisien non-zero: {(lasso.coef_ != 0).sum()}/20")

Perhatikan output: Lasso memilih hanya beberapa fitur (yang `n_informative=5` kita set saat generate data) dan mengabaikan sisanya. Ridge mempertahankan semua 20 fitur dengan koefisien yang lebih kecil. Untuk dataset dengan 1000+ fitur, Lasso jauh lebih **interpretable** dan sering diistimewakan dalam sains. Tapi kelemahannya: kalau fitur-fitur berkorelasi kuat, Lasso cenderung memilih salah satu secara acak — tidak stabil. Untuk itu ada ElasticNet.

Parameter `alpha` di Ridge dan Lasso adalah kekuatan regularisasi: alpha=0 artinya tidak ada regularisasi (sama dengan Linear Regression), alpha=10 artinya penalti sangat kuat. Memilih alpha yang tepat juga pakai cross-validation. Tapi itu topik chapter 04.

---

**Mini-check refleksi 7:**

1. Kapan kamu lebih memilih R² daripada RMSE untuk mengevaluasi model regresi? Apakah R² selalu cukup, atau ada kasus di mana RMSE lebih jujur?
2. Lasso men-zero-kan koefisien fitur yang tidak relevan. Kenapa ini berguna untuk interpretabilitas, dan kenapa Ridge tidak melakukan hal yang sama?
3. Bayangkan kamu melatih model untuk memprediksi harga rumah dan mendapat RMSE = Rp 50 juta. Apakah angka itu "bagus"? Informasi apa lagi yang kamu butuhkan untuk menjawabnya?

---

## Section 8: Mini Project — End-to-End Classification di Dataset Iris

Sekarang saatnya mengikat semua yang sudah kita pelajari. Kita akan melakukan **end-to-end ML pipeline** pada dataset Iris: dari data mentah sampai model akhir yang siap di-pakai. Ini adalah pola yang akan kamu ulang di hampir semua project ML.

Tujuannya bukan hanya mendapat akurasi tinggi, tapi menunjukkan **alur berpikir yang benar**: split yang benar, scaling yang benar, pelatihan yang benar, dan evaluasi yang tidak menipu diri sendiri. Kita akan membandingkan empat model (Logistic Regression, KNN, Decision Tree, Random Forest) menggunakan cross-validation, lalu memilih model terbaik dan mengevaluasinya di test set.

Mari kita mulai dari setup: load data, split, dan definisikan helper untuk evaluasi.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import numpy as np

# 1. Load data
iris = load_iris()
X, y = iris.data, iris.target
target_names = iris.target_names
feature_names = iris.feature_names

# 2. Split stratified 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# 3. Definisikan 4 model sebagai pipeline (scaling otomatis aman)
models = {
    'LogReg'      : Pipeline([('scaler', StandardScaler()),
                                ('clf', LogisticRegression(max_iter=1000, random_state=42))]),
    'KNN (K=5)'   : Pipeline([('scaler', StandardScaler()),
                                ('clf', KNeighborsClassifier(n_neighbors=5))]),
    'DT (depth=3)': Pipeline([('clf', DecisionTreeClassifier(max_depth=3, random_state=42))]),
    'RF (100)'    : Pipeline([('clf', RandomForestClassifier(n_estimators=100, random_state=42))]),
}
print(f"Model yang akan dibanding: {list(models.keys())}")

Perhatikan dua hal penting. Pertama, semua model dibungkus dalam Pipeline, jadi scaling (kalau dibutuhkan) dilakukan sekali di dalam pipeline — tidak akan ada kebocoran. Kedua, **DT dan RF tidak butuh scaling** (mereka tidak menghitung jarak), tapi membungkus mereka dalam Pipeline tidak masalah — hanya saja langkah scaler jadi tidak berguna. Untuk konsistensi, kita tetap pakai Pipeline untuk semua model.

In [ ]:
# 4. Cross-validation 5-fold untuk semua model
results = []
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    results.append({
        'Model'      : name,
        'CV Mean'    : scores.mean(),
        'CV Std'     : scores.std()
    })

results_df = pd.DataFrame(results).sort_values('CV Mean', ascending=False)
print(results_df.round(4).to_string(index=False))

Cross-validation memberikan dua informasi penting: **CV Mean** (akurasi rata-rata) dan **CV Std** (standar deviasi). Mean mengukur seberapa bagus model secara umum, std mengukur seberapa stabil model — std rendah berarti model performanya konsisten di berbagai subset data, std tinggi berarti model sensitif terhadap data yang dipakai.

Untuk Iris, semua model punya CV mean di atas 0.9 — dataset ini memang mudah. Tapi di dunia nyata, perbedaan 0.02 pada CV mean bisa jadi signifikan. Saat memilih model, kita tidak hanya lihat mean tertinggi — kita juga lihat std. Model dengan mean 0.95 dan std 0.02 lebih bisa dipercaya dari model dengan mean 0.96 dan std 0.08.

In [ ]:
# 5. Pilih model terbaik dan evaluasi di test set
best_name = results_df.iloc[0]['Model']
best_model = models[best_name]
print(f"Model terbaik berdasarkan CV: {best_name}")

best_model.fit(X_train, y_train)
test_acc = best_model.score(X_test, y_test)
print(f"Akurasi pada test set: {test_acc:.4f}")

In [ ]:
# 6. Evaluasi lengkap: confusion matrix + classification report
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

y_pred = best_model.predict(X_test)
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=target_names))

Classification report menunjukkan precision, recall, F1 untuk setiap kelas. Untuk kelas 0 (setosa) — precision, recall, F1 = 1.00 artinya model tidak pernah salah mengklasifikasikan bunga setosa. Untuk kelas 1 dan 2, ada beberapa kesalahan (versicolor tertukar virginica, misalnya). Itu wajar — kedua spesies ini memang mirip secara morfologi.

Akurasi agregat 0.97 (29 dari 30 benar) adalah hasil yang sangat bagus. Untuk dataset sederhana seperti Iris, semua model mendekati akurasi 1.0. Tapi di dunia nyata, kamu akan sering mendapat akurasi 0.7-0.85 — dan detail di classification report jadi jauh lebih penting untuk pengambilan keputusan.

In [ ]:
# 7. Visualisasi confusion matrix
fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay.from_estimator(
    best_model, X_test, y_test,
    display_labels=target_names, cmap='Blues', ax=ax, values_format='d'
)
plt.title(f'Confusion Matrix — {best_name}')
plt.tight_layout()
plt.show()

In [ ]:
# 8. Feature importance (kalau model terbaik adalah tree-based)
clf = best_model.named_steps['clf']
if hasattr(clf, 'feature_importances_'):
    imp = pd.DataFrame({
        'fitur'     : feature_names,
        'importance': clf.feature_importances_
    }).sort_values('importance', ascending=False)
    print("Feature importance:")
    print(imp.round(3).to_string(index=False))
    
    plt.figure(figsize=(8, 4))
    plt.barh(imp['fitur'], imp['importance'], color='steelblue')
    plt.gca().invert_yaxis()
    plt.title(f'Feature Importance — {best_name}')
    plt.tight_layout()
    plt.show()
else:
    print(f"{best_name} tidak punya feature_importances_ — lewati visualisasi ini.")

Mari kita rangkum apa yang sudah kita kerjakan. Kita melalui **delapan langkah** yang merupakan pola standar di setiap project ML: (1) load data, (2) split stratified, (3) definisikan model sebagai pipeline, (4) cross-validation untuk perbandingan, (5) pilih model terbaik, (6) fit pada seluruh training set, (7) evaluasi di test set, (8) interpretasi hasil.

Pola ini bisa kamu pakai untuk dataset apa pun — klasifikasi, regresi, dataset besar maupun kecil. Yang berbeda hanya algoritma dan metrik evaluasinya. Setelah chapter ini, cobalah ulangi pola ini untuk **dataset lain** (misal Boston Housing untuk regresi, atau Breast Cancer untuk klasifikasi biner) — kamu akan menemukan bahwa strukturnya tetap sama.

---

**Mini-check refleksi 8 (penutup chapter):**

1. Dari delapan langkah di mini project, langkah mana yang paling berisiko jika dilakukan dengan ceroboh? Jelaskan apa konsekuensinya.
2. Kalau kamu melatih model dengan akurasi 99% di training dan 70% di test, apa yang terjadi? Apa tiga hal yang akan kamu coba untuk memperbaikinya?
3. Chapter 03 fokus pada supervised learning. Sebelum masuk ke deep learning di chapter 04, coba pikirkan: masalah apa yang tidak bisa diselesaikan dengan supervised learning, dan butuh pendekatan lain? Beri satu contoh kasus nyata.

---

## Penutup Chapter

Kamu sudah menyelesaikan chapter paling padat di repo ini. Delapan section yang kita lewati menutupi **siklus lengkap supervised learning** — dari pemahaman konsep (apa itu ML), alur kerja (CRISP-DM, train/test split), fondasi teknis (preprocessing, pipeline), empat algoritma klasifikasi inti (Logistic Regression, KNN, Decision Tree, Random Forest), regresi (Linear, Ridge, Lasso), sampai mini project yang mengikat semuanya.

Yang kamu kuasai sekarang: **membangun, melatih, dan mengevaluasi model ML** dengan scikit-learn. Yang belum kamu kuasai: tuning hyperparameter otomatis (GridSearch), menangani data tidak seimbang, time series, ensemble methods lanjutan (XGBoost, LightGBM), dan deep learning. Semua itu akan kita bahas di chapter 04.

Sebelum lanjut, pastikan kamu sudah mencoba **memodifikasi** kode di notebook ini. Jangan cuma menjalankan — ubah nilai `max_depth`, ganti `n_neighbors`, coba dataset lain. Setiap modifikasi kecil yang kamu lakukan akan membangun intuisi yang jauh lebih kuat daripada membaca saja. Kalau ada bagian yang masih mengganjal, buka `PANDUAN_KODE.md` untuk cheat sheet, atau `praktikum.ipynb` untuk latihan terstruktur.